# exp005 recovery: полный повторный прогон 4B

Смоук уже пройден. `SMOKE=False`. Модель, сплит, batch и lr как в exp005.
Полная validation на шаге 60, далее 200/400/600 и в конце.
Каждый лучший LoRA-адаптер сразу записывается в `best_model`, без копии базы в RAM.
Прогресс печатается каждые 20 шагов. Обучение начинается заново.
Чекпойнт содержит адаптер и validation predictions; optimizer/RNG не сохраняются.


## 1. Установка зависимостей

torch/transformers-стек ставится строго по пинам. numpy/pandas/tqdm переустановка не трогает: ядро Kaggle предзагружает их до старта ячеек, и даунгрейд в том же процессе даёт гибрид версий (память ≠ диск), который ломает импорты scipy/sklearn. Вместо этого они проверяются на совместимый диапазон; фактические версии фиксируются в `environment.json` и в `requirements.txt` репозитория. На чистой сессии Kaggle установка займёт несколько минут (torch скачивается с CUDA-библиотеками).

In [1]:
import importlib.metadata as im
import subprocess
import sys

CORE_LOCK = """torch==2.6.0
transformers==4.51.3
tokenizers==0.21.1
huggingface-hub==0.30.2
safetensors==0.5.3
peft==0.15.2"""

# numpy/pandas/tqdm: ядро предзагружает их до старта ячеек; переустановка другой
# версии в том же процессе даёт гибрид памяти и диска и ломает импорты.
# Поэтому только проверка диапазона; вне диапазона ставится точный fallback-пин.
ENV_RANGES = {'numpy': '>=2.1,<2.3', 'pandas': '>=2.2,<3.2', 'tqdm': '>=4.60'}
ENV_FALLBACK = {'numpy': '2.1.3', 'pandas': '2.3.3', 'tqdm': '4.67.1'}

def core_pins():
    out = {}
    for line in CORE_LOCK.strip().splitlines():
        name, ver = line.split('==')
        out[name] = ver
    return out

def installed_version(dist):
    try:
        return im.version(dist)
    except Exception:
        return None

def loaded_version(name):
    mod = sys.modules.get(name.replace('-', '_'))
    return getattr(mod, '__version__', None) if mod is not None else None

def version_tuple(ver):
    return tuple(int(part) for part in ver.split('.')[:3] if part.isdigit())

def in_range(ver, spec):
    lo, hi = None, None
    for part in spec.split(','):
        part = part.strip()
        if part.startswith('>='):
            lo = tuple(int(x) for x in part[2:].split('.'))
        elif part.startswith('<'):
            hi = tuple(int(x) for x in part[1:].split('.'))
    v = version_tuple(ver)
    if lo is not None and v < lo:
        return False
    if hi is not None and v >= hi:
        return False
    return True

need = {n: v for n, v in core_pins().items() if installed_version(n) != v}
for name, spec in ENV_RANGES.items():
    disk = installed_version(name)
    if disk is None or not in_range(disk, spec):
        loaded = loaded_version(name)
        if loaded is not None:
            raise RuntimeError(
                f'{name}=={loaded} предзагружен ядром и вне допустимого диапазона {spec}; '
                'установка другой версии в этом процессе сломает импорты.'
            )
        need[name] = ENV_FALLBACK[name]

if need:
    print('Установка:', ', '.join(sorted(f'{n}=={v}' for n, v in need.items())))
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-input', '-q']
    cmd += [f'{n}=={v}' for n, v in sorted(need.items())]
    subprocess.check_call(cmd)
else:
    print('Все зависимости уже установлены — pip пропущен.')

# torchvision из образа собран под другой torch и ломает импорт любой модели в
# transformers (loss_utils -> image_utils -> torchvision: 'operator torchvision::nms
# does not exist'). Решению изображения не нужны — сносим, тогда transformers
# проходит по ветке без torchvision (как в локальных проверках).
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision'])
print('torchvision удалён (несовместим с torch 2.6.0, в решении не используется).')

stale = {}
for name in list(core_pins()) + list(ENV_RANGES):
    loaded, disk = loaded_version(name), installed_version(name)
    if loaded is not None and disk is not None and loaded != disk:
        stale[name] = (loaded, disk)
if stale:
    detail = '; '.join(f'{n}: в памяти {old}, на диске {new}' for n, (old, new) in sorted(stale.items()))
    raise RuntimeError(
        'Версия в памяти не совпадает с версией на диске: ' + detail +
        '. Выполни Restart & Run All (полный рестарт сессии) и запусти заново.'
    )

import numpy as np
import pandas as pd
import torch
import transformers

print('python      :', sys.version.split()[0])
print('torch       :', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('pandas      :', pd.__version__, '| numpy:', np.__version__)

Установка: huggingface-hub==0.30.2, peft==0.15.2, safetensors==0.5.3, tokenizers==0.21.1, torch==2.6.0, transformers==4.51.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.6.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.30.2 which is incompatible.
diffusers 0.42.0 requires safetensors>=0.8.0, but you have safetensors 0.5.3 which is incompatible.


torchvision удалён (несовместим с torch 2.6.0, в решении не используется).
python      : 3.13.16
torch       : 2.6.0+cu124 | cuda available: True
transformers: 4.51.3
pandas      : 2.3.3 | numpy: 2.1.3


## 2. Конфигурация, среда и воспроизводимость

Один запуск = одна папка `run_dir` с артефактами. Если `submission.csv` в ней уже лежит, код требует новый `run_name`, чтобы не затирать прошлый эксперимент.

In [2]:
import gc
import hashlib
import html as html_lib
import json
import math
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import random
import re
import subprocess
import sys
import time
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

CFG = {
    'run_name': 'exp005_qwen3_rer4b_lora_seed42_recovery',
    'data_dir': '',            # пусто = автопоиск; иначе путь к папке с train.csv/test.csv
    'out_root': '/kaggle/working',
    'model_id': 'Qwen/Qwen3-Reranker-4B',
    'model_revision': '22e683669bc0f0bd69640a1354a6d0aebcfeede5',
    'seed': 42,                # сплит — фиксирован 42 у всех экспериментов (сравнимость val)
    'train_seed': 42,
    'max_len': 512,            # оверхед chat-шаблона (~80) + query_budget + doc_budget
    'query_budget': 96,
    'doc_budget': 284,
    'doc_char_limit': 2500,    # предобрезка doc-текста по символам (ускоряет токенизацию)
    'batch_size': 4,           # 4B в fp16: VRAM T4 ограничена
    'grad_accum': 8,           # эффективный train batch = 32 (как у остальных)
    'eval_batch_size': 16,
    'epochs': 1,               # пик у всех наших прогонов был в 1-й эпохе; бюджет сессии 12 ч
    'lr': 1e-4,                # LoRA: на 1-2 порядка выше полного файнтюна
    'weight_decay': 0.01,
    'warmup_ratio': 0.10,
    'max_grad_norm': 1.0,
    'eval_every_steps': 200,   # валидация каждые N optimizer-шагов
    'val_fraction': 0.2,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
}

# Смоук уже пройден: этот файл настроен на полный повторный прогон.
# Прошёл — поставить False и запустить полный прогон.
SMOKE = False
SMOKE_STEPS = 60   # 4B медленный: 60 шагов достаточно для гейта
SMOKE_VAL_QUERIES = 300


if not (os.path.isdir(CFG['out_root']) and os.access(CFG['out_root'], os.W_OK)):
    CFG['out_root'] = os.getcwd()
    print('"/kaggle/working" недоступен — out_root =', CFG['out_root'])

run_dir = os.path.join(CFG['out_root'], CFG['run_name'])
if os.path.exists(os.path.join(run_dir, 'submission.csv')):
    raise RuntimeError(f'В {run_dir} уже есть submission.csv — задай новый CFG["run_name"].')
os.makedirs(run_dir, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    cap = torch.cuda.get_device_capability(0)
    amp_dtype = torch.bfloat16 if cap >= (8, 0) else torch.float16
    props = torch.cuda.get_device_properties(0)
    print('GPU      :', torch.cuda.get_device_name(0), '| capability', cap,
          '| VRAM ГБ:', round(props.total_memory / 2**30, 1))
else:
    amp_dtype = torch.float32
    print('CUDA недоступна — обучение пойдёт на CPU (очень медленно).')
print('run_dir  :', run_dir)
print('amp dtype:', amp_dtype)

GPU      : Tesla T4 | capability (7, 5) | VRAM ГБ: 14.6
run_dir  : /kaggle/working/exp005_qwen3_rer4b_lora_seed42_recovery
amp dtype: torch.float16


## 3. Загрузка данных

Автопоиск папки с `train.csv`/`test.csv`, контроль схемы, уникальности `id` и SHA-256 исходных файлов (фиксируются в артефактах запуска).

In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def find_data_dir(cfg):
    candidates = []
    if cfg['data_dir']:
        candidates.append(cfg['data_dir'])
    candidates.append('.')
    root = '/kaggle/input'
    if os.path.isdir(root):
        for dirpath, _, filenames in os.walk(root):
            if 'train.csv' in filenames and 'test.csv' in filenames:
                candidates.append(dirpath)
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'train.csv')) and os.path.isfile(os.path.join(c, 'test.csv')):
            return c
    raise FileNotFoundError('Не найдена папка с train.csv и test.csv — заполни CFG["data_dir"].')

DATA_DIR = find_data_dir(CFG)
train_csv_path = os.path.join(DATA_DIR, 'train.csv')
test_csv_path = os.path.join(DATA_DIR, 'test.csv')
print('DATA_DIR =', DATA_DIR)

train_hash = sha256_file(train_csv_path)
test_hash = sha256_file(test_csv_path)
print('train.csv sha256:', train_hash)
print('test.csv  sha256:', test_hash)

REQUIRED = ['id', 'query_id', 'document_id', 'article_id', 'query', 'title', 'text',
            'passage_index', 'article_passage_count', 'article_char_count', 'ingest_source']

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

assert set(REQUIRED).issubset(train_df.columns), 'train.csv: не хватает колонок'
assert set(REQUIRED).issubset(test_df.columns), 'test.csv: не хватает колонок'
assert not train_df['id'].duplicated().any(), 'train.csv: дубликаты id'
assert not test_df['id'].duplicated().any(), 'test.csv: дубликаты id'

n_train_pos = int(train_df['relevance'].sum())
print(f'train: {len(train_df)} строк, {train_df["query_id"].nunique()} запросов, '
      f'позитивов {n_train_pos} ({n_train_pos / len(train_df):.1%})')
print(f'test : {len(test_df)} строк, {test_df["query_id"].nunique()} запросов')
print('пересечение query_id train/test:', len(set(train_df["query_id"]) & set(test_df["query_id"])))

DATA_DIR = /kaggle/input/datasets/nikitailyutkin/amml-ranking
train.csv sha256: 2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042
test.csv  sha256: 4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20
train: 31530 строк, 3042 запросов, позитивов 8168 (25.9%)
test : 13507 строк, 1305 запросов
пересечение query_id train/test: 0


## 4. Очистка текстов

Тексты содержат HTML-теги и entities, NBSP и неразрывные символы. Очистка: NFC-нормализация → удаление HTML-комментариев → `html.unescape` (до фикспойнта) → удаление известных тегов → схлопывание пробелов. Регистр, пунктуация и математические записи вида `x < 5` сохраняются: шаблон тега требует известного имени тега. Очищаются только `query`, `title`, `text`; метаданные и id не трогаются.

In [4]:
COMMENT_RE = re.compile(r'<!--.*?-->', re.DOTALL)
TAG_NAME = (
    r'p|br|hr|div|span|a|b|strong|i|em|u|s|small|sub|sup|code|pre|blockquote|'
    r'h[1-6]|ul|ol|li|dl|dt|dd|table|thead|tbody|tfoot|tr|td|th|caption|'
    r'figure|figcaption|img|center|font|section|article|header|footer|nav|aside|abbr|cite|mark'
)
TAG_RE = re.compile(r'</?(?:' + TAG_NAME + r')(?:\s[^<>]*)?/?>', re.IGNORECASE)
WS_RE = re.compile(r'\s+')

def clean_text(s):
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFC', s)
    s = s.replace('\u00a0', ' ').replace('\u00ad', '').replace('\u200b', '')
    s = COMMENT_RE.sub(' ', s)
    prev = None
    for _ in range(3):                      # разворачиваем цепочки &amp;amp; до фикспойнта
        s = html_lib.unescape(s)
        if s == prev:
            break
        prev = s
    s = TAG_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

for df in (train_df, test_df):
    for col in ('query', 'title', 'text'):
        df[col + '_c'] = df[col].map(clean_text)

changed = sum(
    ((df[c] != df[c + '_c']) & df[c].notna()).sum()
    for df in (train_df, test_df) for c in ('query', 'title', 'text')
)
print('изменено строк текста:', int(changed))
for c in ('query', 'title', 'text'):
    print(f'пусто после очистки {c}: train {int((train_df[c + "_c"] == "").sum())}, '
          f'test {int((test_df[c + "_c"] == "").sum())}')

mask = (train_df['text'].fillna('') != train_df['text_c']) & (train_df['text'].fillna('') != '')
idx = mask.idxmax()
print('\nПример очистки text до :', repr(str(train_df.loc[idx, 'text'])[:160]))
print('Пример очистки text после:', repr(train_df.loc[idx, 'text_c'][:160]))

изменено строк текста: 31016
пусто после очистки query: train 0, test 0
пусто после очистки title: train 1491, test 737
пусто после очистки text: train 0, test 0

Пример очистки text до : 'Город является крупным железнодорожным узлом Центральной Украины, имеет порт на\xa0Днепре и аэродром. В городе имеется два железнодорожных вокзала: Кременчуг и Крю'
Пример очистки text после: 'Город является крупным железнодорожным узлом Центральной Украины, имеет порт на Днепре и аэродром. В городе имеется два железнодорожных вокзала: Кременчуг и Крю'


## 5. Метрика NDCG@10

Дословная реализация правил соревнования: сортировка по `prediction` по убыванию, при равенстве — по `id` по возрастанию **как строке**; gain `2^relevance − 1`; discount `1/log2(r+1)`; cutoff `min(10, число кандидатов)`; IDCG по всем кандидатам запроса; запросы с IDCG=0 не дают вклада. Ниже — самопроверки на конструктивных случаях.

In [5]:
def ndcg_at_10(df, scores):
    """Средний NDCG@10 по запросам; df — колонки id, query_id, relevance."""
    ids = df['id'].astype(str).to_numpy()
    qids = df['query_id'].to_numpy()
    rel = df['relevance'].astype(float).to_numpy()
    scores = np.asarray(scores, dtype=np.float64)
    # primary: score по убыванию; secondary: id по возрастанию как строке
    order = np.lexsort((ids, -scores))
    tmp = pd.DataFrame({'q': qids[order], 'r': rel[order], 'i': ids[order]})
    total, counted = 0.0, 0
    for _, g in tmp.groupby('q', sort=False):
        rels = g['r'].to_numpy()[:10]
        disc = 1.0 / np.log2(np.arange(len(rels)) + 2.0)
        dcg = float(((2.0 ** rels) - 1.0) @ disc)
        ideal = np.sort(g['r'].to_numpy())[::-1][:10]
        idcg = float(((2.0 ** ideal) - 1.0) @ (1.0 / np.log2(np.arange(len(ideal)) + 2.0)))
        if idcg > 0:
            total += dcg / idcg
            counted += 1
    return total / counted if counted else 0.0

# --- самопроверки ---
m = pd.DataFrame({'id': ['a', 'b', 'c'], 'query_id': [1, 1, 1], 'relevance': [1, 0, 0]})
assert abs(ndcg_at_10(m, [3, 2, 1]) - 1.0) < 1e-12, 'идеальный порядок'
assert abs(ndcg_at_10(m, [0, 1, 0]) - 1.0 / math.log2(3)) < 1e-12, 'позитив на 2-й позиции'

# tie-break: равные скоры → id по возрастанию как строка ('r_001' раньше 'r_002')
m2 = pd.DataFrame({'id': ['r_002', 'r_001'], 'query_id': [1, 1], 'relevance': [1, 0]})
assert abs(ndcg_at_10(m2, [0.5, 0.5]) - 1.0 / math.log2(3)) < 1e-12, 'tie-break по строке id'
assert abs(ndcg_at_10(m2, [0.6, 0.4]) - 1.0) < 1e-12, 'явный порядок перекрывает tie-break'

# запрос с IDCG=0 не даёт вклада в среднее
m3 = pd.DataFrame({
    'id': ['a', 'b', 'c', 'd'],
    'query_id': [1, 1, 2, 2],
    'relevance': [1, 0, 0, 0],
})
assert abs(ndcg_at_10(m3, [1, 0, 0, 0]) - 1.0) < 1e-12, 'исключение IDCG=0'

# cutoff @10: позитив на 11-й позиции не учитывается
m4 = pd.DataFrame({'id': [f'r_{i:03d}' for i in range(12)],
                   'query_id': [1] * 12,
                   'relevance': [0] * 10 + [1, 0]})
assert abs(ndcg_at_10(m4, list(range(12))[::-1])) < 1e-12, 'cutoff @10'

print('Самопроверки метрики: OK')

Самопроверки метрики: OK


## 6. Разделение train 80/20 по запросам

Запросы распределяются детерминированно: сортировка по `SHA-256(seed:query_id)`, первые 20% — validation. Пересечения запросов между fit и val нет, поэтому валидация имитирует закрытые запросы Private-лидерборда. Сплит сохраняется в `split.csv` для сопоставимости будущих экспериментов.

In [6]:
def query_hash(qid, seed):
    return hashlib.sha256(f'{seed}:{qid}'.encode('utf-8')).hexdigest()

all_qids = np.array(sorted(train_df['query_id'].unique()))
hexvals = np.array([int(query_hash(q, CFG['seed'])[:8], 16) for q in all_qids], dtype=np.float64)
is_val = (hexvals / float(2**32)) < CFG['val_fraction']
qid2part = dict(zip(all_qids, np.where(is_val, 'val', 'fit')))

train_df['part'] = train_df['query_id'].map(qid2part)
fit_df = train_df[train_df['part'] == 'fit'].copy().reset_index(drop=True)
val_df = train_df[train_df['part'] == 'val'].copy().reset_index(drop=True)

assert set(fit_df['query_id']).isdisjoint(set(val_df['query_id']))
assert len(fit_df) + len(val_df) == len(train_df)

split_out = pd.DataFrame({'query_id': all_qids, 'part': np.where(is_val, 'val', 'fit')})
split_out.to_csv(os.path.join(run_dir, 'split.csv'), index=False)

for name, part in (('fit', fit_df), ('val', val_df)):
    npos = int(part['relevance'].sum())
    print(f'{name}: {len(part)} строк, {part["query_id"].nunique()} запросов, '
          f'позитивов {npos} ({npos / len(part):.1%}), '
          f'запросов без позитивов: {int((part.groupby("query_id")["relevance"].sum() == 0).sum())}')

print('Константный бейзлайн на val (все скоры 0 → порядок по id):',
      round(ndcg_at_10(val_df, np.zeros(len(val_df))), 4))

fit: 25170 строк, 2428 запросов, позитивов 6458 (25.7%), запросов без позитивов: 0
val: 6360 строк, 614 запросов, позитивов 1710 (26.9%), запросов без позитивов: 0
Константный бейзлайн на val (все скоры 0 → порядок по id): 0.583


## 7. Токенизация: chat-шаблон Qwen3-Reranker

Пара рендерится шаблоном модели: system-блок судьи → `<Instruct>/<Query>/<Document>` →
assistant-блок `<think></think>`; скор читается с последней позиции (следующий токен —
yes/no). Бюджеты по частям: query ≤ 96 токенов, doc ≤ 284 (текст → декод обратно в
строку для шаблона), предобрезка doc 2500 символов; вместе с оверхедом шаблона всё
гарантированно ≤ max_len 512. Паддинг — слева, чтобы последняя позиция была настоящей.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'], revision=CFG['model_revision'])
tokenizer.padding_side = 'left'   # скор на последней позиции — паддинг слева
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD = tokenizer.pad_token_id
YES_ID = tokenizer.convert_tokens_to_ids('yes')
NO_ID = tokenizer.convert_tokens_to_ids('no')
assert YES_ID is not None and NO_ID is not None, 'в словаре токенизатора нет yes/no'
print('yes/no/pad ids:', YES_ID, NO_ID, PAD)

RERANKER_INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'

def build_doc_text(title_c, text_c):
    t = (title_c or '').strip()
    x = (text_c or '').strip()
    if t and x:
        sep = ' ' if t.endswith(('.', '!', '?')) else '. '
        return t + sep + x
    return t or x

def encode_pair(q, doc):
    # бюджет по частям: токенизуем каждую часть, обрезаем до лимита, декодируем
    # обратно в строку для chat-шаблона — суммарная длина гарантирована даже
    # с оверхедом шаблона
    q_ids = tokenizer(q, add_special_tokens=False, truncation=True,
                      max_length=CFG['query_budget'])['input_ids']
    d_ids = tokenizer(doc, add_special_tokens=False, truncation=True,
                      max_length=CFG['doc_budget'])['input_ids']
    messages = [
        {'role': 'system', 'content': RERANKER_INSTRUCTION},
        {'role': 'query', 'content': tokenizer.decode(q_ids)},
        {'role': 'document', 'content': tokenizer.decode(d_ids)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)

def encode_frame(df, desc):
    docs = [build_doc_text(t, x)[:CFG['doc_char_limit']]
            for t, x in zip(df['title_c'], df['text_c'])]
    queries = df['query_c'].tolist()
    out = []
    for q, d in tqdm(list(zip(queries, docs)), total=len(queries), desc=desc):
        out.append(encode_pair(q, d))
    return out

X_fit_ids = encode_frame(fit_df, 'tokenize fit')
X_val_ids = encode_frame(val_df, 'tokenize val')
X_test_ids = encode_frame(test_df, 'tokenize test')

def len_stats(name, seqs):
    lens = np.array([len(s) for s in seqs])
    print(f'{name}: len p50={int(np.percentile(lens, 50))}, p95={int(np.percentile(lens, 95))}, '
          f'max={int(lens.max())}')
    return lens

len_stats('fit', X_fit_ids)
len_stats('val', X_val_ids)
len_stats('test', X_test_ids)
assert max(len(s) for s in X_fit_ids + X_val_ids + X_test_ids) <= CFG['max_len']


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

yes/no/pad ids: 9693 2152 151643


tokenize fit:   0%|          | 0/25170 [00:00<?, ?it/s]

tokenize val:   0%|          | 0/6360 [00:00<?, ?it/s]

tokenize test:   0%|          | 0/13507 [00:00<?, ?it/s]

fit: len p50=345, p95=381, max=427
val: len p50=340, p95=380, max=395
test: len p50=343, p95=381, max=415


## 8. Dataset и DataLoader

Динамический padding по батчу (pad = `<|endoftext|>`), паддинг СЛЕВА — последняя позиция всегда настоящая (там читается скор). Для точной нормировки loss на неполном последнем окне накопления заранее считается план окон (число микробатчей и суммарный размер окна на каждый optimizer-шаг).

In [8]:
class PairDataset(Dataset):
    def __init__(self, ids_list, labels=None):
        self.ids_list = ids_list
        self.labels = labels

    def __len__(self):
        return len(self.ids_list)

    def __getitem__(self, i):
        if self.labels is None:
            return {'ids': self.ids_list[i]}
        return {'ids': self.ids_list[i], 'label': float(self.labels[i])}

def collate(batch):
    maxlen = max(len(b['ids']) for b in batch)
    ids = torch.full((len(batch), maxlen), PAD, dtype=torch.long)
    att = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b['ids'])
        ids[i, maxlen - n:] = torch.tensor(b['ids'], dtype=torch.long)  # паддинг слева
        att[i, maxlen - n:] = 1
    out = {'input_ids': ids, 'attention_mask': att}
    if 'label' in batch[0]:
        out['labels'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

def move_to_device(b):
    return {k: v.to(device, non_blocking=True) for k, v in b.items()}

pin = device.type == 'cuda'
train_loader = DataLoader(PairDataset(X_fit_ids, fit_df['relevance'].to_numpy()),
                          batch_size=CFG['batch_size'], shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=pin)

def window_plan(n_items, batch_size, accum):
    """План optimizer-шагов: (число микробатчей в окне, суммарный размер окна)."""
    sizes = [min(batch_size, n_items - s) for s in range(0, n_items, batch_size)]
    micros, samples, cur = [], [], []
    for s in sizes:
        cur.append(s)
        if len(cur) == accum:
            micros.append(len(cur))
            samples.append(sum(cur))
            cur = []
    if cur:
        micros.append(len(cur))
        samples.append(sum(cur))
    return micros, samples

micros_plan, samples_plan = window_plan(len(X_fit_ids), CFG['batch_size'], CFG['grad_accum'])
print(f'микробатчей/эпоху: {len(X_fit_ids) // CFG["batch_size"] + (len(X_fit_ids) % CFG["batch_size"] > 0)}, '
      f'optimizer-шагов/эпоху: {len(samples_plan)}, окон неполных: {sum(1 for m in micros_plan if m < CFG["grad_accum"])}')

микробатчей/эпоху: 6293, optimizer-шагов/эпоху: 787, окон неполных: 1


## 9. LoRA 4B: обучение с сохранением адаптера на диск

CE по `[no, yes]`, lr=1e-4, 1 эпоха. Лучший checkpoint сохраняется во время обучения.


In [9]:
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], revision=CFG['model_revision'], torch_dtype=torch.float16,  # 4B: base в fp16, адаптер — fp32
)
lora_cfg = LoraConfig(
    r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.config.use_cache = False
model.enable_input_require_grads()   # иначе градиент не проходит checkpointing к LoRA
model.to(device)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()

no_decay = ('bias', 'LayerNorm.weight')
decay_params = [p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)]
nodecay_params = [p for n, p in model.named_parameters()
                  if p.requires_grad and any(nd in n for nd in no_decay)]
optimizer = torch.optim.AdamW(
    [{'params': decay_params, 'weight_decay': CFG['weight_decay']},
     {'params': nodecay_params, 'weight_decay': 0.0}],
    lr=CFG['lr'],
)

steps_per_epoch = len(samples_plan)
total_steps = steps_per_epoch * CFG['epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * CFG['warmup_ratio']),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))
print(f'optimizer-шагов всего: {total_steps}, warmup: {int(total_steps * CFG["warmup_ratio"])}')

if SMOKE:
    total_steps = SMOKE_STEPS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * CFG['warmup_ratio']),
        num_training_steps=total_steps,
    )
    smoke_sig0 = {n: float(p.detach().float().abs().sum().item())
                  for n, p in model.named_parameters() if p.requires_grad}
    smoke_losses = []
    smoke_stop = False

@torch.no_grad()
def predict_ids(ids_list, batch_size):
    """Логиты для списка токенизованных пар; порядок вывода = порядок входа."""
    model.eval()
    loader = DataLoader(PairDataset(ids_list), batch_size=batch_size, shuffle=False,
                        collate_fn=collate, num_workers=0, pin_memory=pin)
    outs = []
    for batch in loader:
        batch = move_to_device(batch)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
            logits = model(input_ids=batch['input_ids'],
                           attention_mask=batch['attention_mask']).logits[:, -1, :]
        outs.append((logits[:, YES_ID] - logits[:, NO_ID]).float().cpu().numpy())
    return np.concatenate(outs)

def eval_val():
    scores = predict_ids(X_val_ids, CFG['eval_batch_size'])
    return float(ndcg_at_10(val_df, scores)), scores

log_path = os.path.join(run_dir, 'training_log.jsonl')
log_f = open(log_path, 'a', encoding='utf-8')

def log_line(record):
    log_f.write(json.dumps(record, ensure_ascii=False) + '\n')
    log_f.flush()

best = {'ndcg': -1.0, 'step': -1}
best_model_dir = os.path.join(run_dir, 'best_model')

def save_best(ndcg, step, scores):
    # Save only the LoRA adapter. Never clone the frozen 4B base into host RAM.
    # Keep the previous best intact until the new directory is fully written.
    import shutil
    import uuid
    candidate = os.path.join(run_dir, 'checkpoint_step_' + str(step) + '_' + uuid.uuid4().hex[:8])
    model.save_pretrained(candidate, safe_serialization=True)
    tokenizer.save_pretrained(candidate)
    predictions = val_df.copy()
    predictions['prediction'] = scores
    predictions.to_csv(os.path.join(candidate, 'validation_predictions.csv'), index=False)
    record = {'ndcg': float(ndcg), 'step': int(step)}
    with open(os.path.join(candidate, 'checkpoint_metrics.json'), 'w') as f:
        json.dump(record, f, indent=2)
    previous = best_model_dir + '_previous'
    if os.path.isdir(previous):
        shutil.rmtree(previous)
    if os.path.isdir(best_model_dir):
        os.replace(best_model_dir, previous)
    os.replace(candidate, best_model_dir)
    best.update(record)
    if os.path.isdir(previous):
        shutil.rmtree(previous)
    print(f'CHECKPOINT SAVED: step {step}, val {ndcg:.4f} -> {best_model_dir}', flush=True)

t_start = time.time()
global_step = 0

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    ep_micros, ep_samples = window_plan(len(X_fit_ids), CFG['batch_size'], CFG['grad_accum'])
    run_loss, run_n = 0.0, 0
    wi, micro_cnt = 0, 0
    for batch in train_loader:
        batch = move_to_device(batch)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
            logits = model(input_ids=batch['input_ids'],
                           attention_mask=batch['attention_mask']).logits
            two = logits[:, -1, :][:, [NO_ID, YES_ID]].float()  # [no, yes] на последнем токене
            loss_sum = F.cross_entropy(two, batch['labels'].long(), reduction='sum')
        scaler.scale(loss_sum / ep_samples[wi]).backward()
        run_loss += float(loss_sum.item())
        run_n += int(batch['labels'].size(0))
        micro_cnt += 1
        if micro_cnt == ep_micros[wi]:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            log_line({'event': 'train', 'step': global_step, 'epoch': epoch,
                      'loss': run_loss / max(run_n, 1),
                      'lr': scheduler.get_last_lr()[0],
                      'elapsed_min': round((time.time() - t_start) / 60, 1)})
            if global_step % 20 == 0:
                print(f'train step {global_step}/{total_steps} | '
                      f'{(time.time() - t_start) / 60:.1f} мин', flush=True)
            if SMOKE:
                smoke_losses.append(run_loss / max(run_n, 1))
                if global_step >= SMOKE_STEPS:
                    smoke_stop = True
                    break  # ближайший цикл — for batch in train_loader
            run_loss, run_n = 0.0, 0
            wi += 1
            micro_cnt = 0
            if global_step == 60 or global_step % CFG['eval_every_steps'] == 0:
                ndcg, checkpoint_scores = eval_val()
                log_line({'event': 'eval', 'step': global_step, 'val_ndcg_at_10': ndcg})
                print(f'step {global_step:5d} | val NDCG@10 = {ndcg:.4f} | '
                      f'{(time.time() - t_start) / 60:.0f} мин')
                if ndcg > best['ndcg']:
                    save_best(ndcg, global_step, checkpoint_scores)
            model.train()
    # валидация в конце эпохи (если последний шаг не только что валидировался)
    if global_step % CFG['eval_every_steps'] != 0:
        ndcg, checkpoint_scores = eval_val()
        log_line({'event': 'eval', 'step': global_step, 'val_ndcg_at_10': ndcg})
        print(f'эпоха {epoch} завершена | step {global_step} | val NDCG@10 = {ndcg:.4f} | '
              f'{(time.time() - t_start) / 60:.0f} мин')
        if ndcg > best['ndcg']:
            save_best(ndcg, global_step, checkpoint_scores)
    if SMOKE and smoke_stop:
        break  # выходим из цикла эпох

    model.train()

assert best['step'] > 0 and os.path.isdir(best_model_dir), 'Нет сохранённого checkpoint'

if SMOKE:
    changed = [n for n, p in model.named_parameters()
               if p.requires_grad
               and float(p.detach().float().abs().sum().item()) != smoke_sig0.get(n, 0.0)]
    first = sum(smoke_losses[:3]) / max(len(smoke_losses[:3]), 1)
    last = sum(smoke_losses[-3:]) / max(len(smoke_losses[-3:]), 1)
    smoke_qids = val_df['query_id'].drop_duplicates().sample(SMOKE_VAL_QUERIES, random_state=0)
    mask = val_df['query_id'].isin(smoke_qids).to_numpy()
    sub_ids = [X_val_ids[i] for i in np.where(mask)[0]]
    smoke_ndcg = ndcg_at_10(val_df[mask], predict_ids(sub_ids, CFG['eval_batch_size']))
    smoke_floor = ndcg_at_10(val_df[mask], np.zeros(int(mask.sum())))
    checks = [
        ('веса (LoRA) изменились', len(changed) > 0),
        (f'loss упал ({first:.4f} -> {last:.4f})', last < first),
        (f'val NDCG@10 подвыборки {smoke_ndcg:.4f} > константного {smoke_floor:.4f}',
         smoke_ndcg > smoke_floor),
    ]
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise RuntimeError(f'СМОУК-ГЕЙТ ПРОВАЛЕН: {failed} — конфиг отбракован, '
                           'полный прогон не запускать')
    raise RuntimeError('Смоук-гейт пройден ✓ — поставь SMOKE=False и запусти полный '
                       'прогон (Save & Run All)')
# The best adapter is already on disk; the next cell reloads it.
log_f.close()
gc.collect()
best_step_global = best['step']
print(f'Лучший чекпойнт: step {best_step_global}, val NDCG@10 = {best["ndcg"]:.4f} → {best_model_dir}')

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.06G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

trainable params: 33,030,144 || all params: 4,054,814,720 || trainable%: 0.8146
optimizer-шагов всего: 787, warmup: 78
train step 20/787 | 9.0 мин
train step 40/787 | 18.3 мин
train step 60/787 | 27.5 мин
step    60 | val NDCG@10 = 0.9408 | 56 мин
CHECKPOINT SAVED: step 60, val 0.9408 -> /kaggle/working/exp005_qwen3_rer4b_lora_seed42_recovery/best_model
train step 80/787 | 65.7 мин
train step 100/787 | 74.9 мин
train step 120/787 | 84.0 мин
train step 140/787 | 93.3 мин
train step 160/787 | 102.6 мин
train step 180/787 | 111.9 мин
train step 200/787 | 121.2 мин
step   200 | val NDCG@10 = 0.9477 | 150 мин
CHECKPOINT SAVED: step 200, val 0.9477 -> /kaggle/working/exp005_qwen3_rer4b_lora_seed42_recovery/best_model
train step 220/787 | 159.3 мин
train step 240/787 | 168.6 мин
train step 260/787 | 177.9 мин
train step 280/787 | 187.1 мин
train step 300/787 | 196.4 мин
train step 320/787 | 205.7 мин
train step 340/787 | 215.0 мин
train step 360/787 | 224.3 мин
train step 380/787 | 233.5 мин


## 10. Финальная валидация лучшей модели

Лучшая модель перечитывается с диска (проверка артефакта), считается validation NDCG@10, константный бейзлайн и разбивка по источникам `ingest_source` (для каждой группы метрика считается по подмножеству кандидатов; запросы без позитивов в подмножестве исключаются автоматически).

In [10]:
# Optimizer parameter groups also hold the model tensors alive.
optimizer.zero_grad(set_to_none=True)
del optimizer, scheduler, scaler, decay_params, nodecay_params, logits, two, loss_sum
# Input-grad hooks/checkpoint closures are released along with the model.
del model
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

from peft import PeftModel

best_model_dir = os.path.join(run_dir, 'best_model')
base = AutoModelForCausalLM.from_pretrained(CFG['model_id'], revision=CFG['model_revision'],
                                            torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, best_model_dir)
model.config.use_cache = False
model.to(device)

val_scores = predict_ids(X_val_ids, CFG['eval_batch_size'])
val_ndcg = float(ndcg_at_10(val_df, val_scores))
const_ndcg = float(ndcg_at_10(val_df, np.zeros(len(val_df))))
print(f'val NDCG@10 (лучшая модель): {val_ndcg:.4f}')
print(f'val NDCG@10 (константный бейзлайн): {const_ndcg:.4f}')

per_source = {}
for src in sorted(val_df['ingest_source'].astype(str).unique()):
    m = (val_df['ingest_source'].astype(str) == src).to_numpy()
    per_source[src] = {
        'rows': int(m.sum()),
        'positive_rate': float(val_df.loc[m, 'relevance'].mean()),
        'ndcg_at_10': float(ndcg_at_10(val_df.loc[m].reset_index(drop=True), val_scores[m])),
    }
print('По источникам:', json.dumps(per_source, ensure_ascii=False, indent=2))

val_pred = val_df[['id', 'query_id', 'document_id', 'article_id', 'ingest_source',
                   'passage_index', 'relevance']].copy()
val_pred['prediction'] = val_scores
val_pred.to_csv(os.path.join(run_dir, 'validation_predictions.csv'), index=False)

metrics = {
    'run_name': CFG['run_name'],
    'best_step': best_step_global,
    'val_ndcg_at_10': val_ndcg,
    'val_constant_ndcg_at_10': const_ndcg,
    'val_rows': int(len(val_df)),
    'val_queries': int(val_df['query_id'].nunique()),
    'val_positives': int(val_df['relevance'].sum()),
    'per_source': per_source,
}
with open(os.path.join(run_dir, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

val NDCG@10 (лучшая модель): 0.9539
val NDCG@10 (константный бейзлайн): 0.5830
По источникам: {
  "S0": {
    "rows": 3173,
    "positive_rate": 0.26284273558146865,
    "ndcg_at_10": 0.9601372877807411
  },
  "S1": {
    "rows": 1990,
    "positive_rate": 0.2884422110552764,
    "ndcg_at_10": 0.973453855355525
  },
  "S2": {
    "rows": 735,
    "positive_rate": 0.2585034013605442,
    "ndcg_at_10": 0.9751828013132151
  },
  "S3": {
    "rows": 462,
    "positive_rate": 0.24242424242424243,
    "ndcg_at_10": 0.9715646682461505
  }
}


## 11. Инференс на test и submission.csv

Все 13 507 тестовых пар получают скор. Проверки: конечность скоров, соответствие множеству `id` теста и `sample_submission.csv` (если найден), порядок строк выравнивается по сэмплу. `submission.csv` пишется в корень output и в папку запуска.

In [11]:
model.eval()
test_scores = predict_ids(X_test_ids, CFG['eval_batch_size'])
assert np.isfinite(test_scores).all(), 'нефининые скоры на test'
assert len(test_scores) == len(test_df)

sub = pd.DataFrame({'id': test_df['id'], 'prediction': test_scores})

sample_path = None
for name in ('sample_submission.csv',):
    p = os.path.join(DATA_DIR, name)
    if os.path.isfile(p):
        sample_path = p
if sample_path:
    sample = pd.read_csv(sample_path)
    assert list(sample.columns) == ['id', 'prediction'], 'неожиданные колонки sample_submission'
    assert set(sample['id']) == set(sub['id']), 'id сэмпла и предсказаний не совпадают'
    sub = sub.set_index('id').loc[sample['id']].reset_index()
    print('sample_submission найден — порядок строк выровнен по нему.')

assert sub['prediction'].notna().all()
sub_path = os.path.join(run_dir, 'submission.csv')
sub.to_csv(sub_path, index=False)
sub.to_csv(os.path.join(CFG['out_root'], 'submission.csv'), index=False)
print('submission.csv:', sub_path)
print(sub.head(3).to_string(index=False))
print('описание prediction:', sub['prediction'].describe().to_dict())

submission.csv: /kaggle/working/exp005_qwen3_rer4b_lora_seed42_recovery/submission.csv
                    id  prediction
r_00100bd94cf8513c6500    1.242188
r_0011d1a81cda1b1c720f    2.234375
r_00135550b108ebd88432   -3.406250
описание prediction: {'count': 13507.0, 'mean': -2.469651222229004, 'std': 3.225053310394287, 'min': -11.0078125, '25%': -4.9375, '50%': -2.9375, '75%': -0.15625, 'max': 6.328125}


## 12. Артефакты запуска и итог

`environment.json` фиксирует среду и хэши данных, `run_summary.txt` — человекочитаемую сводку. Значения из `metrics.json` переносятся в журнал экспериментов `README.md` вручную после запуска.

In [12]:
def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

import importlib.metadata
import platform

env = {
    'run_name': CFG['run_name'],
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'machine': platform.machine(),
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
    'gpu_capability': list(torch.cuda.get_device_capability(0)) if device.type == 'cuda' else None,
    'cuda_runtime': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if device.type == 'cuda' else None,
    'torch': pkg_version('torch'),
    'transformers': pkg_version('transformers'),
    'peft': pkg_version('peft'),
    'tokenizers': pkg_version('tokenizers'),
    'pandas': pkg_version('pandas'),
    'numpy': pkg_version('numpy'),
    'amp_dtype': str(amp_dtype),
    'model_id': CFG['model_id'],
    'model_revision': CFG['model_revision'],
    'seed': CFG['seed'],
    'train_seed': CFG.get('train_seed', CFG['seed']),
    'lora_r': CFG.get('lora_r'),
    'max_len': CFG['max_len'],
    'files': {'train.csv': train_hash, 'test.csv': test_hash},
    'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(run_dir, 'environment.json'), 'w', encoding='utf-8') as f:
    json.dump(env, f, ensure_ascii=False, indent=2)

elapsed_min = (time.time() - t_start) / 60
summary = f"""Запуск: {CFG['run_name']}
Модель: {CFG['model_id']} @ {CFG['model_revision']}
Вход: query -> title+text, max_len={CFG['max_len']} (query<={CFG['query_budget']}, doc<={CFG['doc_budget']}), очистка HTML/entities/NBSP
Split: 80/20 по query_id, seed={CFG['seed']} (SHA-256), fit={len(fit_df)} строк / {fit_df['query_id'].nunique()} запросов, val={len(val_df)} строк / {val_df['query_id'].nunique()} запросов
Обучение: LoRA CE (yes/no), r={CFG.get('lora_r')}, lr={CFG['lr']}, effective batch={CFG['batch_size'] * CFG['grad_accum']}, epochs={CFG['epochs']}, precision={amp_dtype}
Лучший чекпойнт: step {best_step_global}, val NDCG@10 = {val_ndcg:.4f}
Константный бейзлайн на val: {const_ndcg:.4f}
По источникам: {json.dumps({k: round(v['ndcg_at_10'], 4) for k, v in per_source.items()}, ensure_ascii=False)}
Время: {elapsed_min:.0f} мин
ОС/GPU: {env['platform']} / {env['gpu']}
Артефакты: {run_dir}
Submission: {os.path.join(CFG['out_root'], 'submission.csv')}
"""
with open(os.path.join(run_dir, 'run_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(summary)
log_f.close()
print(summary)

Запуск: exp005_qwen3_rer4b_lora_seed42_recovery
Модель: Qwen/Qwen3-Reranker-4B @ 22e683669bc0f0bd69640a1354a6d0aebcfeede5
Вход: query -> title+text, max_len=512 (query<=96, doc<=284), очистка HTML/entities/NBSP
Split: 80/20 по query_id, seed=42 (SHA-256), fit=25170 строк / 2428 запросов, val=6360 строк / 614 запросов
Обучение: LoRA CE (yes/no), r=16, lr=0.0001, effective batch=32, epochs=1, precision=torch.float16
Лучший чекпойнт: step 787, val NDCG@10 = 0.9539
Константный бейзлайн на val: 0.5830
По источникам: {"S0": 0.9601, "S1": 0.9735, "S2": 0.9752, "S3": 0.9716}
Время: 599 мин
ОС/GPU: Linux-6.18.55+-x86_64-with-glibc2.39 / Tesla T4
Артефакты: /kaggle/working/exp005_qwen3_rer4b_lora_seed42_recovery
Submission: /kaggle/working/submission.csv

